# Model Performance Evaluation

What this notebook does:

1. It provide the opportunity to validate the scores that each lighter-weight approach has given to each fact-candidate pair

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from rapidfuzz import fuzz
from Levenshtein import ratio, setratio, seqratio, jaro, jaro_winkler
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax
from sentence_transformers import util

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model_paraph = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")
model_cross = CrossEncoder('cross-encoder/nli-deberta-v3-base')
model_cross_advanc = CrossEncoder("MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli")

## Edit-distance

In [ ]:
fact = "Here comes the fact"
sentence = "Here comes the sentence"

lev_ratio = ratio(fact, sentence)

rapfuz_token_sort = fuzz.token_sort_ratio(fact, sentence)

print(f" Levenshtein Ratio: {lev_ratio:.4f}")
print()
print(f"RapidFuzz Token_sort: {rapfuz_token_sort:.3f}")

## Embedding + Paraphrase

In [ ]:
fact = "Here comes the fact"
sentence = "Here comes the sentence"
emb_fact = model.encode(fact)
emb_sentence = model.encode(sentence)

paraph_fact = model_paraph.encode(fact)
paraph_sentence = model_paraph.encode(sentence)

emb_score = util.cos_sim(emb_fact, emb_sentence).item()
paraph_score = util.cos_sim(paraph_fact, paraph_sentence).item()

print(f" Embedding : {emb_score:.4f}")
print()
print(f" Paraphrase : {paraph_score:.4f}")

## NLI

In [ ]:
print(model_cross.config.id2label)
print(model_cross_advanc.config.id2label)

In [ ]:
# small NLI model
labels = ["contradiction", "entailment", "neutral"]  # 0, 1, 2


fact     = "Here comes the fact"
sentence = "Here comes the sentence"

logits = model_cross.predict([(sentence, fact)])[0]
probs  = softmax(logits)

contradict_score, entail_score, neutral_score = probs
print(dict(zip(labels, np.round(probs, 4))))
print("label:", labels[int(np.argmax(probs))])

In [ ]:
# large NLI model
labels_adv = ["entailment", "neutral", "contradicted"]
fact     = "Here comes the fact"
sentence = "Here comes the sentence"

logits_adv = model_cross_advanc.predict([(sentence, fact)])[0]
probs_adv  = softmax(logits_adv)

entail_score,  neutral_score, contradict_score = probs_adv
print(dict(zip(labels_adv, np.round(probs_adv, 4))))
print("label:", labels_adv[int(np.argmax(probs_adv))])